# Tome 5, Chapter 3: Dirichlet characters and L-functions (Python companion)

Exercises first, solutions at the end. We build every Dirichlet character from the structure of $(\mathbb Z/q\mathbb Z)^*$, test orthogonality, find conductors and count primitive characters, compute Gauss sums, check Dirichlet's formula for $L(1,\chi)$, meet class numbers, and finish with the Jacobi symbol and the Solovay–Strassen primality test. Requirements: mpmath.

*Before each solution, predict what the output will look like.*

## Exercises

**P3.1 (★★) All the characters.** Write a function returning all characters modulo $q$, using that $(\mathbb Z/q\mathbb Z)^*$ is a product of cyclic groups: one for each odd prime power (generated by a primitive root), and $\langle-1\rangle\times\langle5\rangle$ for $2^e$, $e\ge3$. Print the tables for $q=5,8,12$ and compare with Exercise 3.1.

**P3.2 (★) Orthogonality and conductors.** Check multiplicativity and both orthogonality relations (Proposition 3.1) for all $q\le60$. Compute conductors (Proposition 3.4) and check the count of primitive characters of Exercise 3.2.

**P3.3 (★) Gauss sums.** Check Theorem 3.5 for all primitive characters with $q\le60$, see that it fails for an imprimitive one, and compute $\tau$ for the Legendre symbol modulo $5,7,11,13,1019$ (Tome 4, Theorem 8.14).

**P3.4 (★★) $L(1,\chi)$.** Implement Dirichlet's formula (Theorem 3.10) and compare it with an independent computation, $L(1,\chi)=-\frac1q\sum_a\chi(a)\psi(\frac aq)$ with the digamma function $\psi$, for all primitive $\chi$ modulo $q\le40$. Which is the smallest $|L(1,\chi)|$? Then watch $\prod_\chi L(\sigma,\chi)$ as $\sigma\to1^+$ (Lemma 3.7).

**P3.5 (★) Class numbers.** For primes $p\equiv3\pmod4$ below $400$, compute $h(-p)=-\frac1p\sum_{a<p}\bigl(\frac ap\bigr)a$, check that it is a positive integer (Exercise 3.6) and that $\sum_{n<p/2}\bigl(\frac np\bigr)=(2-(\frac2p))h(-p)$. For which $p$ is $h(-p)=1$?

**P3.6 (★★) CS application: the Solovay–Strassen test.** Compute the Jacobi symbol $\bigl(\frac an\bigr)$ for odd $n$ by quadratic reciprocity, without factoring $n$. For prime $n$, Euler's criterion says $a^{(n-1)/2}\equiv\bigl(\frac an\bigr)\pmod n$. Count the \emph{Euler liars} (the $a$ satisfying it) for some Carmichael numbers, and use the criterion as a randomised primality test.

## Solutions

### P3.1

In [ ]:
import math, cmath, random
import mpmath as mp

def factor(n):
    f, p = {}, 2
    while p * p <= n:
        while n % p == 0:
            f[p] = f.get(p, 0) + 1
            n //= p
        p += 1
    if n > 1:
        f[n] = f.get(n, 0) + 1
    return f

def characters(q):
    """All Dirichlet characters mod q, as lists chi[0..q-1] of complex numbers.
    (Z/qZ)^* is a product of cyclic groups: one per odd prime power (a primitive root), and <-1> x <5> for 2^e, e >= 3."""
    gens = []   # (modulus m, generator g, order) for the cyclic factors, with the CRT lift of g
    for p, e in factor(q).items():
        m = p**e
        if p == 2:
            if e == 1:
                continue
            gens.append((m, m - 1, 2))
            if e >= 3:
                gens.append((m, 5, 2**(e - 2)))
        else:
            phi = m - m // p
            g = next(g for g in range(2, m) if math.gcd(g, p) == 1 and all(pow(g, phi // r, m) != 1 for r in factor(phi)))
            gens.append((m, g, phi))
    # discrete logs of every unit along each generator, computed by walking the cyclic factors
    units = [a for a in range(q) if math.gcd(a, q) == 1]
    logs = {a: [] for a in units}
    for m, g, order in gens:
        if m == q or len(gens) == 1 and m == q:
            pass
        table, x = {}, 1
        for k in range(order):
            table[x] = k
            x = x * g % m
        for a in units:
            r = a % m
            if m % 2 == 0 and order == 2 and g == m - 1:     # the factor <-1> mod 2^e
                logs[a].append(0 if r % 4 == 1 else 1)
            elif m % 2 == 0:                                  # the factor <5> mod 2^e: a or -a is a power of 5
                r = r if r % 4 == 1 else (-r) % m
                logs[a].append(table[r])
            else:
                logs[a].append(table[r])
    chars = []
    from itertools import product
    for ks in product(*[range(o) for _, _, o in gens]):
        chi = [0j] * q
        for a in units:
            chi[a] = cmath.exp(2j * math.pi * sum(k * l / o for k, l, (_, _, o) in zip(ks, logs[a], gens)))
        chars.append(chi)
    return chars

for q in (5, 8, 12):
    print(f"q = {q}:")
    for chi in characters(q):
        print("   ", [complex(round(c.real, 6), round(c.imag, 6)) if abs(c.imag) > 1e-9 else round(c.real) for c in chi])

The tables match Exercise 3.1: modulo $5$ two complex characters with $\chi(2)=\pm i$; modulo $8$ and $12$ all characters are real, because both groups are $(\mathbb Z/2\mathbb Z)^2$. The decomposition of $(\mathbb Z/2^e\mathbb Z)^*$ as $\langle-1\rangle\times\langle5\rangle$ is the one non-cyclic case: every odd $a$ is $\pm5^k$ modulo $2^e$.

### P3.2

In [ ]:
def is_multiplicative(chi, q):
    return all(abs(chi[a * b % q] - chi[a] * chi[b]) < 1e-9 for a in range(q) for b in range(q))

def conductor(chi, q):
    for d in sorted(d for d in range(1, q + 1) if q % d == 0):
        if all(abs(chi[n] - 1) < 1e-9 for n in range(q) if math.gcd(n, q) == 1 and n % d == 1 % d):
            return d

ok_mult = ok_orth = True
for q in range(2, 61):
    chs = characters(q)
    phi = sum(1 for a in range(q) if math.gcd(a, q) == 1)
    assert len(chs) == phi
    ok_mult &= all(is_multiplicative(c, q) for c in chs)
    # orthogonality: sum over n, and sum over chi
    ok_orth &= all(abs(sum(c)) < 1e-6 for c in chs[1:]) and abs(sum(chs[0]) - phi) < 1e-6
    ok_orth &= all(abs(sum(c[a] for c in chs) - (phi if a == 1 % q else 0)) < 1e-6 for a in range(q))
print("characters multiplicative:", ok_mult, "  orthogonality relations:", ok_orth)

def phistar(q):
    r = 1
    for p, e in factor(q).items():
        r *= p - 2 if e == 1 else p**e * (p - 1)**2 // p**2
    return r

bad = [q for q in range(2, 61) if sum(conductor(c, q) == q for c in characters(q)) != phistar(q)]
print("number of primitive characters = (mu * phi)(q) for all q <= 60:", not bad)
print("conductors mod 12:", [conductor(c, 12) for c in characters(12)], "  mod 16:", sorted(conductor(c, 16) for c in characters(16)))

Everything checks. Modulo $12$ the conductors are $1,3,4,12$, as in Exercise 3.1; modulo $16$, half the characters are primitive, in line with $\varphi^*(16)=16(1-\frac12)^2=4$.

### P3.3

In [ ]:
def e(x):
    return cmath.exp(2j * math.pi * x)

def gauss_sum(chi, q, n=1):
    return sum(chi[a] * e(a * n / q) for a in range(q))

worst = 0
for q in range(3, 61):
    for chi in characters(q):
        if conductor(chi, q) != q:
            continue
        t = gauss_sum(chi, q)
        worst = max(worst, abs(abs(t)**2 - q))
        for n in range(q):
            worst = max(worst, abs(gauss_sum(chi, q, n) - chi[n].conjugate() * t))
print("primitive characters, q <= 60: max error in sum chi(a)e(an/q) = conj(chi(n)) tau and |tau|^2 = q:", f"{worst:.1e}")
# a non-primitive example: chi_0 chi_{-4} mod 12
chi12 = next(c for c in characters(12) if conductor(c, 12) == 4)
print("induced from conductor 4, mod 12: |tau|^2 =", round(abs(gauss_sum(chi12, 12))**2, 6), " (not 12)")
for p in (5, 7, 11, 13, 1019):
    leg = [0] + [1 if pow(a, (p - 1) // 2, p) == 1 else -1 for a in range(1, p)]
    t = gauss_sum(leg, p)
    print(f"p = {p:4d}: tau = {t.real:+.6f} {t.imag:+.6f} i   sqrt(p) = {math.sqrt(p):.6f}")

For primitive characters the identity $\sum_a\chi(a)e(\frac{an}q)=\overline{\chi(n)}\tau(\chi)$ holds to rounding error, including $n$ not prime to $q$. For the character modulo $12$ induced from $\chi_{-4}$, $|\tau|^2=4$: primitivity matters. The Legendre symbol has $\tau=\sqrt p$ for $p\equiv1\pmod4$ and $i\sqrt p$ for $p\equiv3\pmod4$, exactly as Gauss proved.

### P3.4

In [ ]:
mp.mp.dps = 20
def L_hurwitz(s, chi, q):
    """L(s, chi) = q^(-s) sum_a chi(a) zeta(s, a/q); at s = 1 (chi non-principal) use -(1/q) sum_a chi(a) digamma(a/q)."""
    if s == 1:
        return complex(-sum(mp.mpc(chi[a]) * mp.digamma(mp.mpf(a) / q) for a in range(1, q)) / q)
    return complex(mp.mpf(q)**(-s) * sum(mp.mpc(chi[a]) * mp.zeta(s, mp.mpf(a) / q) for a in range(1, q + 1) if chi[a % q] != 0))

def L1_formula(chi, q):
    """Theorem: Dirichlet's formula for L(1, chi), chi primitive mod q > 1."""
    cb = [c.conjugate() for c in chi]
    tb = gauss_sum(cb, q)
    if abs(chi[q - 1] + 1) < 1e-9:   # odd
        return -1j * math.pi / (q * tb) * sum(cb[a] * a for a in range(1, q))
    return -1 / tb * sum(cb[a] * math.log(math.sin(math.pi * a / q)) for a in range(1, q))

worst, smallest = 0, (9, None)
for q in range(3, 41):
    for chi in characters(q):
        if conductor(chi, q) != q:
            continue
        v = L1_formula(chi, q)
        ref = L_hurwitz(1, chi, q)
        worst = max(worst, abs(v - ref))
        if abs(v) < smallest[0]:
            smallest = (abs(v), q)
print(f"Dirichlet's formula vs mpmath, all primitive chi mod q <= 40: max error {worst:.1e}")
print(f"smallest |L(1, chi)| among them: {smallest[0]:.4f} (q = {smallest[1]})")
print("L(1, chi_-3) =", L1_formula([0, 1, -1], 3).real, " pi/(3 sqrt 3) =", math.pi / (3 * math.sqrt(3)))
print("L(1, chi_5)  =", L1_formula([0, 1, -1, -1, 1], 5).real, " (2/sqrt5) log golden =", 2 / math.sqrt(5) * math.log((1 + math.sqrt(5)) / 2))
# Lemma: the product of all L(sigma, chi) is >= 1, and blows up like 1/(sigma - 1)
for q in (5, 7, 12):
    chs = characters(q)
    for s in (2, 1.1, 1.01):
        P = 1
        for chi in chs:
            P *= L_hurwitz(s, chi, q)
        print(f"q = {q:2d}, sigma = {s:5}: product = {P.real:9.4f} {P.imag:+.0e}i")

Dirichlet's finite formula agrees with the digamma computation to $10^{-13}$ for every primitive character up to $40$. Among them $|L(1,\chi)|\ge0.41$ (the minimum at $q=19$), far from $0$; small values of $L(1,\chi)$ for real $\chi$ would mean exceptionally small class numbers, and how small they can be is the open problem of Siegel zeros. The product over all characters is real, at least $1$, and grows like $\frac c{\sigma-1}$: the pole of $\chi_0$ times the non-zero values $L(1,\chi)$.

### P3.5

In [ ]:
def legendre(a, p):
    a %= p
    return 0 if a == 0 else (1 if pow(a, (p - 1) // 2, p) == 1 else -1)

hs = []
for p in range(7, 400):
    if all(p % d for d in range(2, int(p**0.5) + 1)) and p % 4 == 3:
        s = sum(a * legendre(a, p) for a in range(1, p))
        assert s < 0 and s % p == 0
        half = sum(legendre(n, p) for n in range(1, p // 2 + 1))
        h = -s // p
        assert half == (2 - legendre(2, p)) * h
        hs.append((p, h))
print("class numbers h(-p), p = 3 mod 4:", hs[:25])
print("h(-p) = 1 for p in", [p for p, h in hs if h == 1])
p = 1019
print(f"h(-{p}) =", -sum(a * legendre(a, p) for a in range(1, p)) // p, f"  L(1) = pi h / sqrt(p) = {math.pi * 13 / math.sqrt(p):.6f}",
      f"  series: {L_hurwitz(1, [legendre(a, p) for a in range(p)], p).real:.6f}")

Every $h(-p)$ is a positive integer, so the non-residues always outweigh the residues, and the half-sum identity holds. The primes with $h(-p)=1$ are $7,11,19,43,67,163$: together with $3$ these are the primes $p\equiv3\pmod4$ for which $\mathbb Z[\frac{1+\sqrt{-p}}2]$ has unique factorisation. That the list is complete is the Baker–Heegner–Stark theorem; $163$ is behind the near-integer $e^{\pi\sqrt{163}}$ (Tome 12). For $p=1019$, $h=13$ and $L(1,\chi)=\frac{\pi h}{\sqrt p}=1.2794$, matching the series.

### P3.6

In [ ]:
def jacobi(a, n):
    """Jacobi symbol (a/n) for odd n > 0, by quadratic reciprocity: O(log n) steps, no factoring."""
    a %= n
    result = 1
    while a:
        while a % 2 == 0:
            a //= 2
            if n % 8 in (3, 5):
                result = -result
        a, n = n, a
        if a % 4 == 3 and n % 4 == 3:
            result = -result
        a %= n
    return result if n == 1 else 0

assert all(jacobi(a, p) == legendre(a, p) for p in (3, 5, 7, 101, 1019) for a in range(3 * p))

def solovay_strassen(n, rounds=30):
    if n < 3 or n % 2 == 0:
        return n == 2
    for _ in range(rounds):
        a = random.randrange(2, n - 1)
        j = jacobi(a, n)
        if j == 0 or pow(a, (n - 1) // 2, n) != j % n:
            return False
    return True

def euler_liars(n):
    return sum(1 for a in range(1, n) if math.gcd(a, n) == 1 and pow(a, (n - 1) // 2, n) == jacobi(a, n) % n)

for n in (561, 1105, 1729, 2465, 15841, 29341):
    phi = sum(1 for a in range(1, n) if math.gcd(a, n) == 1)
    print(f"n = {n:6d}: Euler liars {euler_liars(n):5d} of phi(n) = {phi:6d}  (ratio {euler_liars(n) / phi:.3f})")
random.seed(5)
N = 2**127 - 1
print("2^127 - 1 prime by Solovay-Strassen:", solovay_strassen(N), "  2^128 + 1:", solovay_strassen(2**128 + 1))

The Jacobi symbol algorithm is Euclid's algorithm with signs from reciprocity: $O(\log n)$ steps and no factoring. For composite $n$, the Euler liars form a proper subgroup of $(\mathbb Z/n\mathbb Z)^*$, so at most half the units lie, as the table shows (exactly $\frac14$ or $\frac12$ here). Each round of Solovay–Strassen therefore catches a composite with probability at least $\frac12$, and $30$ rounds leave an error probability below $10^{-9}$. Unlike Fermat's test, Carmichael numbers cannot fool it. Miller–Rabin (Chapter 1's notebook) is stronger still, with at most $\frac14$ liars, and replaced Solovay–Strassen in practice.